In [ ]:
# import needed libraries
import os
import joblib
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

# move root folder if inside notebooks folder
if os.getcwd().endswith("notebooks") or os.path.exists("../models"):
    os.chdir("..")

# create models folder if missing
os.makedirs("models", exist_ok=True)

# create dummy model if missing
if not os.path.exists("models/fraud_model.pkl"):
    X_dummy = np.random.rand(100, 379)
    y_dummy = np.random.randint(0, 2, size=100)
    model = lgb.LGBMClassifier(verbose=-1)
    model.fit(X_dummy, y_dummy)
    joblib.dump(model, "models/fraud_model.pkl")

# load saved model
model = joblib.load("models/fraud_model.pkl")

# print short status
print("Model loaded")

Model loaded


In [3]:
# write api script file for backend
with open("api.py", "w") as f:
    f.write('''import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="IEEE CIS Fraud Detection API", description="Real time fraud prediction API for IEEE CIS transaction data", version="1.0.0")
FRAUD_THRESHOLD = 0.77
try:
    model = joblib.load("models/fraud_model.pkl")
except Exception as e:
    model = None

class Transaction(BaseModel):
    features: list[float]

@app.get("/")
def root():
    return {"status": "running", "model_loaded": model is not None}

@app.post("/predict")
def predict(data: Transaction):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    input_array = np.array(data.features).reshape(1, -1)
    raw_output = model.predict(input_array)
    if hasattr(model, "predict_proba"):
        prediction = int(raw_output[0])
        proba = float(model.predict_proba(input_array)[0][1])
    elif 0 <= raw_output[0] <= 1 and not isinstance(raw_output[0], (int, np.integer)):
        proba = float(raw_output[0])
        prediction = 1 if proba >= FRAUD_THRESHOLD else 0
    else:
        prediction = int(raw_output[0])
        proba = None
    result = {"fraud": prediction}
    if proba is not None:
        result["probability"] = round(proba, 6)
        result["threshold"] = FRAUD_THRESHOLD
    return result
''')

# print short status
print("API file created")

API file created


In [1]:
# write streamlit app script
app_code = "import joblib\nimport numpy as np\nimport streamlit as st\n\n# page layout set logo\nst.set_page_config(\n    page_title=\"Team 5 IEEE Fraud Detection\",\n    page_icon=\"https://cdn-icons-png.flaticon.com/512/2092/2092663.png\",\n    layout=\"wide\"\n)\n\n# css light dark mode\nst.markdown('''\n<style>\n    .main-header {font-size: 38px; color: #FF4B4B; font-weight: bold;}\n    .sub-header {font-size: 20px; opacity: 0.8;}\n    .card {\n        background: rgba(128, 128, 128, 0.1);\n        padding: 20px;\n        border-radius: 10px;\n        border-left: 5px solid #FF4B4B;\n        margin-bottom: 10px;\n        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.05);\n        min-height: 210px;\n        display: flex;\n        flex-direction: column;\n    }\n    .team-name {font-size: 22px; font-weight: bold; color: #0083B8;}\n    .role-text {font-size: 16px; font-weight: 600; opacity: 0.9;}\n    .desc-text {font-size: 14px; opacity: 0.8; line-height: 1.5;}\n</style>\n''', unsafe_allow_html=True)\n\n# sidebar nav brand\nst.sidebar.title(\"Team 5 Portal\")\nst.sidebar.markdown(\"### NTI IEEE Fraud Detection\")\npage = st.sidebar.radio(\"Navigation\", [\"Home & Team\", \"Fraud Detector\", \"Project Info\"])\n\n\n# load model cache error\n@st.cache_resource\ndef load_model():\n    try:\n        return joblib.load(\"models/fraud_model.pkl\")\n    except Exception as e:\n        st.error(f\"Failed to load model {e}\")\n        return None\n\n\nmodel = load_model()\n\n# feature names list 379 features\nFEATURE_NAMES = [\n    \"TransactionDT\", \"TransactionAmt\",\n    \"ProductCD_H\", \"ProductCD_R\", \"ProductCD_S\", \"ProductCD_W\",\n    \"card1\", \"card2\", \"card3\",\n    \"card4_discover\", \"card4_mastercard\", \"card4_visa\",\n    \"card5\",\n    \"card6_credit\", \"card6_debit\", \"card6_debit or credit\",\n    \"addr1\", \"addr2\", \"dist1\",\n    \"P_emaildomain\", \"R_emaildomain\",\n    *[f\"C{i}\" for i in range(1, 15)],\n    \"D1\", \"D2\", \"D3\", \"D4\", \"D5\", \"D10\", \"D11\", \"D15\",\n    \"M1\", \"M2\", \"M3\", \"M4_M1\", \"M4_M2\", \"M5\", \"M6\", \"M7\", \"M8\", \"M9\",\n    *[f\"V{i}\" for i in range(1, 138)],\n    *[f\"V{i}\" for i in range(167, 322)],\n    \"id_01\", \"id_02\", \"id_05\", \"id_06\", \"id_11\",\n    \"id_12\", \"id_13\", \"id_15\", \"id_16\", \"id_17\", \"id_19\", \"id_20\",\n    \"id_28\", \"id_29\", \"id_31\", \"id_35\", \"id_36\", \"id_37\", \"id_38\",\n    \"DeviceType_mobile\", \"DeviceInfo\",\n    \"TransactionHour\", \"TransactionDay\",\n    \"card1_amt_mean\", \"card1_amt_std\", \"TransactionAmt_to_mean_card1\",\n    \"uid_count\", \"uid_amt_mean\", \"uid_amt_std\", \"TransactionAmt_to_mean_uid\",\n    \"uid2_count\", \"uid2_amt_mean\", \"TransactionAmt_to_mean_uid2\",\n    \"time_since_last_uid_txn\",\n]\n\nassert len(FEATURE_NAMES) == 379, f\"Expected 379 features got {len(FEATURE_NAMES)}\"\n\n# default feature values\nDEFAULTS = {\n    \"TransactionDT\": 5000000.0,\n    \"TransactionAmt\": 68.5,\n    \"ProductCD_H\": 0, \"ProductCD_R\": 0, \"ProductCD_S\": 0, \"ProductCD_W\": 1,\n    \"card1\": 10000.0, \"card2\": 321.0, \"card3\": 150.0,\n    \"card4_discover\": 0, \"card4_mastercard\": 0, \"card4_visa\": 1,\n    \"card5\": 226.0,\n    \"card6_credit\": 0, \"card6_debit\": 1, \"card6_debit or credit\": 0,\n    \"addr1\": 299.0, \"addr2\": 87.0, \"dist1\": 0.0,\n    \"P_emaildomain\": 100000.0, \"R_emaildomain\": 0.0,\n    **{f\"C{i}\": 1.0 for i in range(1, 15)},\n    \"D1\": 14.0, \"D2\": 14.0, \"D3\": 0.0, \"D4\": 0.0, \"D5\": 0.0,\n    \"D10\": 0.0, \"D11\": 0.0, \"D15\": 0.0,\n    \"M1\": 0, \"M2\": 0, \"M3\": 0, \"M4_M1\": 0, \"M4_M2\": 0,\n    \"M5\": 0, \"M6\": 0, \"M7\": 0, \"M8\": 0, \"M9\": 0,\n    **{f\"V{i}\": 0.0 for i in range(1, 138)},\n    **{f\"V{i}\": 0.0 for i in range(167, 322)},\n    \"id_01\": 0.0, \"id_02\": 0.0, \"id_05\": 0.0, \"id_06\": 0.0, \"id_11\": 0.0,\n    \"id_12\": 0, \"id_13\": 0.0, \"id_15\": 0, \"id_16\": 0,\n    \"id_17\": 0.0, \"id_19\": 0.0, \"id_20\": 0.0,\n    \"id_28\": 0, \"id_29\": 0, \"id_31\": 0.0, \"id_35\": 0, \"id_36\": 0, \"id_37\": 0, \"id_38\": 0,\n    \"DeviceType_mobile\": 0, \"DeviceInfo\": 0.0,\n    \"TransactionHour\": 12, \"TransactionDay\": 3,\n    \"card1_amt_mean\": 100.0, \"card1_amt_std\": 80.0, \"TransactionAmt_to_mean_card1\": 1.0,\n    \"uid_count\": 3.0, \"uid_amt_mean\": 100.0, \"uid_amt_std\": 50.0, \"TransactionAmt_to_mean_uid\": 1.0,\n    \"uid2_count\": 2.0, \"uid2_amt_mean\": 100.0, \"TransactionAmt_to_mean_uid2\": 1.0,\n    \"time_since_last_uid_txn\": -1.0,\n}\n\n# threshold setting\nFRAUD_THRESHOLD = 0.77\n\n\n# helper binary toggle\ndef _binary_toggle(label, key, default=0):\n    return st.selectbox(label, options=[0, 1], index=default, key=key)\n\n\n# helper float input\ndef _float_slider(label, key, default=0.0, min_val=-1.0, max_val=1000.0, step=0.1):\n    return st.number_input(label, min_value=min_val, max_value=max_val,\n                           value=float(default), step=step, key=key)\n\n\n# helper int input\ndef _int_slider(label, key, default=0, min_val=0, max_val=100000):\n    return st.number_input(label, min_value=min_val, max_value=max_val,\n                           value=int(default), step=1, key=key)\n\n\n# page 1 home & team\nif page == \"Home & Team\":\n    st.markdown('<p class=\"main-header\">IEEE CIS Fraud Detection System</p>', unsafe_allow_html=True)\n    st.markdown('<p class=\"sub-header\">Advanced Machine Learning System to Detect Financial Transaction Fraud</p>', unsafe_allow_html=True)\n    st.markdown('<p class=\"sub-header\">NTI Graduation Project</p>', unsafe_allow_html=True)\n    st.write(\"---\")\n\n    st.header(\"Meet Team 5 Members\")\n    col1, col2, col3 = st.columns(3)\n\n    with col1:\n        st.markdown('<div class=\"card\"><p class=\"team-name\">Mohamed Ahdy</p><p class=\"role-text\">Data Engineering & EDA Lead</p><p class=\"desc-text\">Responsible for data cleaning preprocessing missing values & feature engineering</p></div>', unsafe_allow_html=True)\n\n    with col2:\n        st.markdown('<div class=\"card\"><p class=\"team-name\">Abdelrahman Ramadan</p><p class=\"role-text\">AI Modeling & Tuning Lead</p><p class=\"desc-text\">Responsible for handling imbalanced data training baseline models & hyperparameter tuning</p></div>', unsafe_allow_html=True)\n\n    with col3:\n        st.markdown('<div class=\"card\"><p class=\"team-name\">Alfarouq Ibrahim</p><p class=\"role-text\">Deployment & Product Lead</p><p class=\"desc-text\">Responsible for model serialization FastAPI APIs Streamlit portal & GitHub production</p></div>', unsafe_allow_html=True)\n\n# page 2 fraud detector\nelif page == \"Fraud Detector\":\n    st.markdown('<p class=\"main-header\">Real Time Transaction Analysis</p>', unsafe_allow_html=True)\n    st.write(\"Adjust transaction features below using expandable sections click Analyze Transaction\")\n    st.write(\"All fields default value typical legitimate transaction\")\n\n    values = dict(DEFAULTS)\n\n    # section 1 transaction core\n    with st.expander(\"Transaction Core\", expanded=False):\n        st.caption(\"Basic transaction timestamp & amount\")\n        c1, c2 = st.columns(2)\n        with c1:\n            values[\"TransactionDT\"] = _float_slider(\"TransactionDT (seconds since reference)\", \"TransactionDT\",\n                                                     default=5000000.0, min_val=0.0, max_val=20000000.0, step=1000.0)\n        with c2:\n            values[\"TransactionAmt\"] = _float_slider(\"TransactionAmt ($)\", \"TransactionAmt\",\n                                                      default=68.5, min_val=0.0, max_val=50000.0, step=0.5)\n\n    # section 2 product code\n    with st.expander(\"Product Code (One Hot Encoded)\", expanded=False):\n        st.caption(\"Select product code W most common One hot encoded drop first C baseline\")\n        pc = st.radio(\"Product Code\", [\"C (baseline)\", \"H\", \"R\", \"S\", \"W\"], index=4,\n                       key=\"product_code_radio\", horizontal=True)\n        values[\"ProductCD_H\"] = 1 if pc == \"H\" else 0\n        values[\"ProductCD_R\"] = 1 if pc == \"R\" else 0\n        values[\"ProductCD_S\"] = 1 if pc == \"S\" else 0\n        values[\"ProductCD_W\"] = 1 if pc == \"W\" else 0\n\n    # section 3 card info\n    with st.expander(\"Card Information\", expanded=False):\n        st.caption(\"Payment card details billing address & distance\")\n        c1, c2, c3 = st.columns(3)\n        with c1:\n            values[\"card1\"] = _float_slider(\"card1\", \"card1\", 10000.0, 0.0, 20000.0, 1.0)\n            values[\"card2\"] = _float_slider(\"card2\", \"card2\", 321.0, 0.0, 600.0, 1.0)\n            values[\"card3\"] = _float_slider(\"card3\", \"card3\", 150.0, 0.0, 300.0, 1.0)\n            values[\"card5\"] = _float_slider(\"card5\", \"card5\", 226.0, 0.0, 500.0, 1.0)\n        with c2:\n            card4 = st.radio(\"card4 (Card Network)\", [\"american express (baseline)\", \"discover\", \"mastercard\", \"visa\"],\n                              index=3, key=\"card4_radio\")\n            values[\"card4_discover\"] = 1 if card4 == \"discover\" else 0\n            values[\"card4_mastercard\"] = 1 if card4 == \"mastercard\" else 0\n            values[\"card4_visa\"] = 1 if card4 == \"visa\" else 0\n\n            card6 = st.radio(\"card6 (Card Type)\", [\"charge card (baseline)\", \"credit\", \"debit\", \"debit or credit\"],\n                              index=2, key=\"card6_radio\")\n            values[\"card6_credit\"] = 1 if card6 == \"credit\" else 0\n            values[\"card6_debit\"] = 1 if card6 == \"debit\" else 0\n            values[\"card6_debit or credit\"] = 1 if card6 == \"debit or credit\" else 0\n        with c3:\n            values[\"addr1\"] = _float_slider(\"addr1 (billing region)\", \"addr1\", 299.0, 0.0, 500.0, 1.0)\n            values[\"addr2\"] = _float_slider(\"addr2 (billing country)\", \"addr2\", 87.0, 0.0, 102.0, 1.0)\n            values[\"dist1\"] = _float_slider(\"dist1 (distance)\", \"dist1\", 0.0, 0.0, 10000.0, 1.0)\n\n    # section 4 email domains\n    with st.expander(\"Email Domains (Frequency Encoded)\", expanded=False):\n        st.caption(\"Frequency encoded email domain counts from train data\")\n        c1, c2 = st.columns(2)\n        with c1:\n            values[\"P_emaildomain\"] = _float_slider(\"P_emaildomain (purchaser)\", \"P_emaildomain\",\n                                                      100000.0, 0.0, 300000.0, 100.0)\n        with c2:\n            values[\"R_emaildomain\"] = _float_slider(\"R_emaildomain (recipient)\", \"R_emaildomain\",\n                                                      0.0, 0.0, 300000.0, 100.0)\n\n    # section 5 count features C1-C14\n    with st.expander(\"Count Features (C1-C14)\", expanded=False):\n        st.caption(\"Transaction count features measuring grouping aggregations\")\n        cols = st.columns(4)\n        for idx, i in enumerate(range(1, 15)):\n            with cols[idx % 4]:\n                values[f\"C{i}\"] = _float_slider(f\"C{i}\", f\"C{i}\", 1.0, 0.0, 5000.0, 1.0)\n\n    # section 6 time delta features D\n    with st.expander(\"Time Delta Features (D)\", expanded=False):\n        st.caption(\"Time deltas between transaction events\")\n        d_features = [\"D1\", \"D2\", \"D3\", \"D4\", \"D5\", \"D10\", \"D11\", \"D15\"]\n        d_defaults = {\"D1\": 14.0, \"D2\": 14.0, \"D3\": 0.0, \"D4\": 0.0, \"D5\": 0.0,\n                      \"D10\": 0.0, \"D11\": 0.0, \"D15\": 0.0}\n        cols = st.columns(4)\n        for idx, d in enumerate(d_features):\n            with cols[idx % 4]:\n                values[d] = _float_slider(d, d, d_defaults[d], -1.0, 1000.0, 1.0)\n\n    # section 7 match features M\n    with st.expander(\"Match Features (M1-M9)\", expanded=False):\n        st.caption(\"Binary match indicators name address email M4 one hot encoded M0 baseline\")\n        cols = st.columns(5)\n        binary_m = [\"M1\", \"M2\", \"M3\", \"M5\", \"M6\", \"M7\", \"M8\", \"M9\"]\n        for idx, m in enumerate(binary_m):\n            with cols[idx % 5]:\n                values[m] = _binary_toggle(f\"{m} (T=1 / F=0)\", f\"m_{m}\")\n        with cols[3]:\n            m4 = st.radio(\"M4\", [\"M0 (baseline)\", \"M1\", \"M2\"], index=0, key=\"m4_radio\")\n            values[\"M4_M1\"] = 1 if m4 == \"M1\" else 0\n            values[\"M4_M2\"] = 1 if m4 == \"M2\" else 0\n\n    # section 8 vesta features V1-V137\n    with st.expander(\"Vesta Features V1-V137\", expanded=False):\n        st.caption(\"Anonymous Vesta payment features default zero\")\n        cols = st.columns(5)\n        for idx, i in enumerate(range(1, 138)):\n            with cols[idx % 5]:\n                values[f\"V{i}\"] = _float_slider(f\"V{i}\", f\"V{i}\", 0.0, -5.0, 5000.0, 0.1)\n\n    # section 9 vesta features V167-V321\n    with st.expander(\"Vesta Features V167-V321\", expanded=False):\n        st.caption(\"Anonymous Vesta payment features second block default zero\")\n        cols = st.columns(5)\n        for idx, i in enumerate(range(167, 322)):\n            with cols[idx % 5]:\n                values[f\"V{i}\"] = _float_slider(f\"V{i}\", f\"V{i}\", 0.0, -5.0, 5000.0, 0.1)\n\n    # section 10 identity features\n    with st.expander(\"Identity Features\", expanded=False):\n        st.caption(\"Device & network identity info\")\n        numeric_ids = [\"id_01\", \"id_02\", \"id_05\", \"id_06\", \"id_11\", \"id_13\", \"id_17\", \"id_19\", \"id_20\"]\n        cols = st.columns(4)\n        for idx, fid in enumerate(numeric_ids):\n            with cols[idx % 4]:\n                values[fid] = _float_slider(fid, fid, 0.0, -100.0, 100000.0, 1.0)\n        st.write(\"---\")\n        st.caption(\"Binary identity features Found 1 NotFound 0\")\n        binary_ids = [\"id_12\", \"id_15\", \"id_16\", \"id_28\", \"id_29\", \"id_35\", \"id_36\", \"id_37\", \"id_38\"]\n        cols = st.columns(5)\n        for idx, fid in enumerate(binary_ids):\n            with cols[idx % 5]:\n                values[fid] = _binary_toggle(fid, f\"id_{fid}\")\n        st.write(\"---\")\n        st.caption(\"Label encoded identity feature\")\n        values[\"id_31\"] = _float_slider(\"id_31 (browser device label encoded)\", \"id_31\", 0.0, 0.0, 200.0, 1.0)\n\n    # section 11 device features\n    with st.expander(\"Device Features\", expanded=False):\n        st.caption(\"Device type & info frequency encoded\")\n        c1, c2 = st.columns(2)\n        with c1:\n            values[\"DeviceType_mobile\"] = _binary_toggle(\"DeviceType (0=desktop / 1=mobile)\", \"DeviceType_mobile\")\n        with c2:\n            values[\"DeviceInfo\"] = _float_slider(\"DeviceInfo (frequency encoded)\", \"DeviceInfo\",\n                                                  0.0, 0.0, 100000.0, 1.0)\n\n    # section 12 engineered time features\n    with st.expander(\"Engineered Time Features\", expanded=False):\n        st.caption(\"Derived from TransactionDT\")\n        c1, c2 = st.columns(2)\n        with c1:\n            values[\"TransactionHour\"] = _int_slider(\"TransactionHour (0-23)\", \"TransactionHour\", 12, 0, 23)\n        with c2:\n            values[\"TransactionDay\"] = _int_slider(\"TransactionDay (0=Mon 6=Sun)\", \"TransactionDay\", 3, 0, 6)\n\n    # section 13 engineered card aggregation\n    with st.expander(\"Engineered Card1 Aggregation\", expanded=False):\n        st.caption(\"Stats TransactionAmt grouped card1\")\n        c1, c2, c3 = st.columns(3)\n        with c1:\n            values[\"card1_amt_mean\"] = _float_slider(\"card1_amt_mean\", \"card1_amt_mean\",\n                                                      100.0, 0.0, 10000.0, 1.0)\n        with c2:\n            values[\"card1_amt_std\"] = _float_slider(\"card1_amt_std\", \"card1_amt_std\",\n                                                     80.0, 0.0, 5000.0, 1.0)\n        with c3:\n            values[\"TransactionAmt_to_mean_card1\"] = _float_slider(\"TransactionAmt / card1_amt_mean\",\n                                                                     \"TransactionAmt_to_mean_card1\",\n                                                                     1.0, 0.0, 200.0, 0.01)\n\n    # section 14 engineered uid aggregation\n    with st.expander(\"Engineered UID Aggregation\", expanded=False):\n        st.caption(\"UID card1 addr1 D1 Aggregation stats\")\n        c1, c2, c3, c4 = st.columns(4)\n        with c1:\n            values[\"uid_count\"] = _float_slider(\"uid_count\", \"uid_count\", 3.0, 0.0, 5000.0, 1.0)\n        with c2:\n            values[\"uid_amt_mean\"] = _float_slider(\"uid_amt_mean\", \"uid_amt_mean\", 100.0, 0.0, 10000.0, 1.0)\n        with c3:\n            values[\"uid_amt_std\"] = _float_slider(\"uid_amt_std\", \"uid_amt_std\", 50.0, 0.0, 5000.0, 1.0)\n        with c4:\n            values[\"TransactionAmt_to_mean_uid\"] = _float_slider(\"TransactionAmt / uid_amt_mean\",\n                                                                   \"TransactionAmt_to_mean_uid\",\n                                                                   1.0, 0.0, 200.0, 0.01)\n\n    # section 15 engineered uid2 velocity\n    with st.expander(\"Engineered UID2 & Velocity\", expanded=False):\n        st.caption(\"UID2 card1 card2 addr1 P_emaildomain velocity time since last\")\n        c1, c2, c3, c4 = st.columns(4)\n        with c1:\n            values[\"uid2_count\"] = _float_slider(\"uid2_count\", \"uid2_count\", 2.0, 0.0, 5000.0, 1.0)\n        with c2:\n            values[\"uid2_amt_mean\"] = _float_slider(\"uid2_amt_mean\", \"uid2_amt_mean\", 100.0, 0.0, 10000.0, 1.0)\n        with c3:\n            values[\"TransactionAmt_to_mean_uid2\"] = _float_slider(\"TransactionAmt / uid2_amt_mean\",\n                                                                    \"TransactionAmt_to_mean_uid2\",\n                                                                    1.0, 0.0, 200.0, 0.01)\n        with c4:\n            values[\"time_since_last_uid_txn\"] = _float_slider(\"time_since_last_uid_txn (sec -1=first)\",\n                                                                \"time_since_last_uid_txn\",\n                                                                -1.0, -1.0, 1000000.0, 100.0)\n\n    # predict button\n    st.write(\"---\")\n    col1, col2 = st.columns([1, 4])\n    with col1:\n        predict_btn = st.button(\"Analyze Transaction\", type=\"primary\", use_container_width=True)\n\n    if predict_btn:\n        if model is None:\n            st.error(\"Model not loaded Please check model file exist\")\n        else:\n            try:\n                feature_array = np.array([[values[f] for f in FEATURE_NAMES]])\n\n                n_expected = getattr(model, \"n_features_in_\", getattr(model, \"n_features_\", len(FEATURE_NAMES)))\n                if n_expected != len(FEATURE_NAMES):\n                    st.warning(\n                        f\"Feature Mismatch Notice current model expect {n_expected} feature type {type(model).__name__} but pipeline give {len(FEATURE_NAMES)} feature please update model file\"\n                    )\n                else:\n                    raw_output = model.predict(feature_array)\n\n                    if hasattr(model, \"predict_proba\"):\n                        prediction = int(raw_output[0])\n                        proba = float(model.predict_proba(feature_array)[0][1])\n                    elif 0 <= raw_output[0] <= 1 and not isinstance(raw_output[0], (int, np.integer)):\n                        proba = float(raw_output[0])\n                        prediction = 1 if proba >= FRAUD_THRESHOLD else 0\n                    else:\n                        prediction = int(raw_output[0])\n                        proba = None\n\n                    st.write(\"---\")\n                    if prediction == 1:\n                        st.error(\"ALERT FRAUDULENT TRANSACTION DETECTED\")\n                        if proba is not None:\n                            st.metric(\"Fraud Probability\", f\"{proba:.2%}\", delta=f\"Threshold {FRAUD_THRESHOLD:.0%}\")\n                    else:\n                        st.success(\"STATUS NORMAL LEGITIMATE TRANSACTION\")\n                        if proba is not None:\n                            st.metric(\"Fraud Probability\", f\"{proba:.2%}\", delta=f\"Threshold {FRAUD_THRESHOLD:.0%}\",\n                                      delta_color=\"off\")\n            except Exception as e:\n                st.error(f\"Error analyze data {str(e)}\")\n\n# page 3 project info\nelif page == \"Project Info\":\n    st.markdown('<p class=\"main-header\">Project Architecture</p>', unsafe_allow_html=True)\n\n    st.info(\"Dataset IEEE CIS Fraud Detection Kaggle\")\n    st.link_button(\"Dataset Link\", \"https://www.kaggle.com/competitions/ieee-fraud-detection\", type=\"primary\")\n    st.write(\"### Workflow Stages\")\n    st.write(\"1. **Data Understanding & Cleaning**: Removing outliers & handling missing identities\")\n    st.write(\"2. **Feature Engineering**: Creating transactional aggregations & encoding categoricals\")\n    st.write(\"3. **Model Training**: Comparing XGBoost LightGBM & Random Forest with SMOTE\")\n    st.write(\"4. **Realtime Deployment**: RESTful API via FastAPI & interactive UI via Streamlit\")\n\n    st.write(\"### Model Details\")\n    if model is not None:\n        st.write(f\"- **Model type**: `{type(model).__name__}`\")\n        n_features = getattr(model, \"n_features_in_\", getattr(model, \"n_features_\", \"N/A\"))\n        st.write(f\"- **Expected features**: {n_features}\")\n        st.write(f\"- **Fraud threshold**: {FRAUD_THRESHOLD}\")\n    else:\n        st.warning(\"Model not loaded\")\n"

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

# print short status
print("App created")

App created


In [5]:
# generate requirements file for deployment
with open("requirements.txt", "w") as f:
    f.write("fastapi\nuvicorn\npydantic\nstreamlit\nscikit-learn\nlightgbm\npandas\nnumpy\njoblib\n")

# print short status
print("Requirements saved")

Requirements saved


In [6]:
# test prediction with dummy array & check output
n_feat = getattr(model, "n_features_in_", getattr(model, "n_features_", 379))
test_features = np.random.rand(1, n_feat)
pred = model.predict(test_features)

# print short status
print("Test pred", int(pred[0]))

Test pred 0
